<a href="https://colab.research.google.com/github/janani26121992/AI-Projects/blob/main/AI_LSTM_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Necessary Libraries**

In [3]:
import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import word_tokenize

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, LSTM, Embedding, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# **Data Gathering**
"I used the requests library to fetch the Shakespeare dataset directly from GitHub using a URL. Then I converted the response into text format and previewed the first 500 characters."

In [5]:
pip install datasets

In [13]:
import requests

url = "https://www.gutenberg.org/files/11/11-0.txt"
text_data = requests.get(url).text

with open("story.txt", "w", encoding="utf-8") as f:
    f.write(text_data)
print("Dataset downloaded and saved!")

Dataset downloaded and saved!


In [14]:
with open("story.txt", "r", encoding="utf-8") as f:
    text_data = f.read()

print(text_data[:500])  # preview


*** START OF THE PROJECT GUTENBERG EBOOK 11 ***

[Illustration]




Alice’s Adventures in Wonderland

by Lewis Carroll

THE MILLENNIUM FULCRUM EDITION 3.0

Contents

 CHAPTER I.     Down the Rabbit-Hole
 CHAPTER II.    The Pool of Tears
 CHAPTER III.   A Caucus-Race and a Long Tale
 CHAPTER IV.    The Rabbit Sends in a Little Bill
 CHAPTER V.     Advice from a Caterpillar
 CHAPTER VI.    Pig and Pepper
 CHAPTER VII.   A Mad Tea-Party
 CHAPTER VIII.  The Queen’s Croquet-Ground
 CHAPTER IX.    The


In [15]:
import re
text_data = text_data.lower()
text_data = re.sub(r'[^a-zA-Z\s]', '', text_data)

In [16]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text_data])

total_words = len(tokenizer.word_index) + 1
print("Vocabulary Size:", total_words)

# Reverse mapping: index -> word
index_word = {i: word for word, i in tokenizer.word_index.items()}

Vocabulary Size: 2763


In [18]:
token_list = tokenizer.texts_to_sequences([text_data])[0]

input_sequences = []

# Create sequences of 6 words
# First 5 words = input, 6th word = target
for i in range(5, len(token_list)):
    n_gram_sequence = token_list[i-5:i+1]
    input_sequences.append(n_gram_sequence)

print(input_sequences[:5])

[[1485, 7, 1, 1061, 1062, 1063], [7, 1, 1061, 1062, 1063, 1486], [1, 1061, 1062, 1063, 1486, 275], [1061, 1062, 1063, 1486, 275, 527], [1062, 1063, 1486, 275, 527, 11]]


In [19]:
max_sequence_length = 6

input_sequences = np.array(input_sequences)

X = input_sequences[:, :-1] # all sequences except last word/index
y = input_sequences[:, -1] # only last word/index

'''
[7, 121, 2, 1, 634, 158]
x= [7, 121, 2, 1, 634 ]
y = [158]
'''

# Convert target to one-hot encoding
y = to_categorical(y, num_classes=total_words)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (26471, 5)
y shape: (26471, 2763)


In [20]:
model = Sequential()

# Correct input shape = sequence length - 1
model.add(Input(shape=(max_sequence_length - 1,))) # 5

# Embedding layer
model.add(Embedding(input_dim=total_words, output_dim=128))

# First LSTM layer
model.add(LSTM(150, return_sequences=True, dropout=0.2))

# Second LSTM layer
model.add(LSTM(100, dropout=0.2))

# Hidden Dense layer
model.add(Dense(100, activation='relu'))

# Output layer
model.add(Dense(total_words, activation='softmax')) #units=6032

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 5, 128)         │       353,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 5, 150)         │       167,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 100)            │       100,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2763)           │       279,063 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 910,627 (3.47 MB)

 Trainable params: 910,627 (3.47 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
early_stop = EarlyStopping(monitor='loss',patience=3,restore_best_weights=True)

history = model.fit(X,y,epochs=30,batch_size=32,verbose=1,callbacks=[early_stop])

In [ ]:
model.save("TextGenerationModelbookcorpus.keras")

In [ ]:
def sample_with_temperature(preds, temperature=0.8, top_k=5):
    preds = np.asarray(preds).astype("float64")
    # [0.87,0.09,0.56,0.44,0.37,.............,0.89,0.32,...]

    # Select top k probabilities
    top_indices = np.argsort(preds)[-top_k:]
    # argsort : [0.09,0.32,0.37,0.44,0.56,0.87,0.89,........]
    # index of top k proabilities : [index]
    top_probs = preds[top_indices]
    #top k probs

    # Apply temperature scaling
    top_probs = np.log(top_probs + 1e-10) / temperature
    exp_probs = np.exp(top_probs)
    top_probs = exp_probs / np.sum(exp_probs)

    return np.random.choice(top_indices, p=top_probs)

In [ ]:
def generate_text(seed_text, next_words=20):
    output_text = seed_text
    generated_words = []

    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([output_text])[0]

        token_list = pad_sequences(
            [token_list],
            maxlen=max_sequence_length - 1,
            padding='pre'
        ) # [0,0,0,189,45]

        predicted_probs = model.predict(token_list, verbose=0)[0] # [[0.89,0.07,.....,]] = [0.89,0.07,.....,]

        predicted_index = sample_with_temperature(
            predicted_probs,
            temperature=0.8,
            top_k=5
        )

        next_word = index_word.get(predicted_index, "")

        # avoid immediate repetition
        if next_word in generated_words[-3:]:
            continue

        generated_words.append(next_word)
        output_text += " " + next_word

    return output_text

In [ ]:

print(generate_text("So shaken", next_words=15))
